# Feature Engineering — TFM Granada

Pipeline limpio de ingeniería de características para el modelo de scoring cooperativo de municipios.

**Scores calculados:** `coopscore_v3/v4/v5/v6`, `IVU`, `IVEC`, `perfil_municipio`

## 1. Imports y rutas

In [1]:
import unicodedata

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
import shap

BASE_DIR       = Path.cwd().parent
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_EXTERNAL  = BASE_DIR / "data" / "external"

c:\Users\pepetorres\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Funciones auxiliares

In [2]:
def limpiar_texto(texto: str) -> str:
    """Normaliza texto: minúsculas, strip y elimina tildes/diacríticos."""
    texto = str(texto).strip().lower()
    return ''.join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


def norm_minmax(series: pd.Series) -> np.ndarray:
    """MinMax normalización sobre una serie, devuelve array 1-D."""
    return MinMaxScaler().fit_transform(series.values.reshape(-1, 1)).ravel()

## 3. Carga de datos

In [3]:
df         = pd.read_csv(DATA_PROCESSED / "dataset_modelado_v2.csv")
distancias = pd.read_csv(DATA_EXTERNAL  / "distancia_pueblos.csv", sep=";")
precios    = pd.read_csv(DATA_EXTERNAL  / "precios_municipios_granada_2026.csv")

print(f"df: {df.shape}  |  distancias: {distancias.shape}  |  precios: {precios.shape}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\pepetorres\\data\\processed\\dataset_modelado_v2.csv'

## 4. Merge de distancias (capital y costa)

In [ ]:
# --- Normalizar claves de unión ---
CORRECCIONES = {
    "gabias, las": "las gabias",
    "valle, el":   "el valle",
    "malaha, la":  "la malaha",
    "calahorra, la": "la calahorra",
    "peza, la":    "la peza",
}

df["municipio_merge"]         = df["municipio"].apply(limpiar_texto).replace(CORRECCIONES)
distancias["municipio_merge"] = distancias["localidad"].apply(limpiar_texto)

distancias = distancias.rename(columns={
    "Distancia en Kms Capital":   "distancia_capital_km",
    "Distancia en kms a la costa": "distancia_costa_km",
})

# --- Merge limpio (eliminar columnas previas si existen) ---
df_v3 = (
    df
    .drop(columns=["distancia_capital_km", "distancia_costa_km"], errors="ignore")
    .merge(
        distancias[["municipio_merge", "distancia_capital_km", "distancia_costa_km"]],
        on="municipio_merge",
        how="left",
    )
)

# --- Diagnóstico ---
n_sin_capital = df_v3["distancia_capital_km"].isna().sum()
n_sin_costa   = df_v3["distancia_costa_km"].isna().sum()
print(f"Sin distancia capital: {n_sin_capital}  |  Sin distancia costa: {n_sin_costa}")
if n_sin_capital:
    display(df_v3.loc[df_v3["distancia_capital_km"].isna(), ["municipio", "municipio_merge"]])

## 5. Merge de precios por municipio

In [ ]:
precios["municipio_merge"] = precios["municipio"].apply(limpiar_texto)

df_v3 = df_v3.merge(
    precios[["municipio_merge", "precio_venta_eur_m2"]],
    on="municipio_merge",
    how="left",
)

print(f"Municipios con precio: {df_v3['precio_venta_eur_m2'].notna().sum()} / {len(df_v3)}")

## 6. Imputación de valores faltantes (mediana)

In [ ]:
for col in ["distancia_capital_km", "distancia_costa_km"]:
    df_v3[col] = df_v3[col].fillna(df_v3[col].median())

# Imputar precio antes de normalizar
df_v3["precio_venta_eur_m2"] = df_v3["precio_venta_eur_m2"].fillna(
    df_v3["precio_venta_eur_m2"].median()
)

## 7. Cálculo de scores normalizados

In [ ]:
# Scores de distancia (transformación inversa)
df_v3["score_capital"] = 1 / (1 + df_v3["distancia_capital_km"])
df_v3["score_costa"]   = 1 / (1 + df_v3["distancia_costa_km"])

# Normalización MinMax
df_v3["score_capital_norm"]  = norm_minmax(df_v3["score_capital"])
df_v3["score_costa_norm"]    = norm_minmax(df_v3["score_costa"])
df_v3["score_precio"]        = norm_minmax(df_v3["precio_venta_eur_m2"])
df_v3["score_n_poligonos"]   = norm_minmax(df_v3["n_poligonos"])

## 8. Score litoral

In [ ]:
MUNICIPIOS_COSTA = {
    "Almuñécar", "Salobreña", "Motril", "Polopos",
    "Gualchos",  "Rubite",    "Sorvilán", "Lújar",
}
MUNICIPIOS_INFLUENCIA_ALTA = {
    "Vélez de Benaudalla", "Molvízar", "Jete", "Otívar",
    "Ítrabo",              "Lentegí",  "Los Guájares",
}

df_v3["score_litoral"] = (
    df_v3["municipio"]
    .map(lambda m: 1.00 if m in MUNICIPIOS_COSTA
                   else 0.75 if m in MUNICIPIOS_INFLUENCIA_ALTA
                   else 0.0)
)

## 9. CoopaScores (v3 → v6)

In [ ]:
# --- v3: distancias + demografía + suelo ---
df_v3["coopscore_v3"] = (
    df_v3["score_superficie"]    * 0.30 +
    df_v3["score_log_poblacion"] * 0.25 +
    df_v3["score_suelo_hab"]     * 0.20 +
    df_v3["score_capital_norm"]  * 0.15 +
    df_v3["score_costa_norm"]    * 0.10
) * 100

# --- v4: penalización municipios pequeños (<5 000 hab.) ---
df_v3["factor_poblacion"] = np.where(df_v3["poblacion_2025"] < 5_000, 0.75, 1.0)
df_v3["coopscore_v4"] = df_v3["coopscore_v3"] * df_v3["factor_poblacion"]

# --- v5: incorpora precio de venta ---
df_v3["coopscore_v5"] = (
    df_v3["score_superficie"]    * 0.25 +
    df_v3["score_log_poblacion"] * 0.20 +
    df_v3["score_suelo_hab"]     * 0.15 +
    df_v3["score_capital_norm"]  * 0.15 +
    df_v3["score_costa_norm"]    * 0.10 +
    df_v3["score_precio"]        * 0.15
) * 100

# --- v6: litoral explícito, sin precio ---
df_v3["coopscore_v6"] = (
    df_v3["score_superficie"]    * 0.25 +
    df_v3["score_log_poblacion"] * 0.20 +
    df_v3["score_suelo_hab"]     * 0.15 +
    df_v3["score_capital_norm"]  * 0.25 +
    df_v3["score_litoral"]       * 0.15
) * 100

print("Scores calculados:", [c for c in df_v3.columns if c.startswith("coopscore")])

## 10. Índices IVU e IVEC

In [ ]:
# IVU — Índice de Valor Urbanístico
df_v3["IVU"] = (
    df_v3["score_superficie"]          * 0.50 +
    df_v3["score_suelo_hab"]           * 0.30 +
    (1 - df_v3["score_n_poligonos"])   * 0.20
) * 100

# IVEC — Índice de Valor Económico-Comercial
df_v3["IVEC"] = (
    df_v3["score_precio"]        * 0.40 +
    df_v3["score_log_poblacion"] * 0.30 +
    df_v3["score_capital_norm"]  * 0.20 +
    df_v3["score_litoral"]       * 0.10
) * 100

## 11. Clasificación por perfil de municipio

In [ ]:
def clasificar_perfil(row):
    if   row["IVEC"] >= 50 and row["IVU"] >= 50: return "Alta Prioridad"
    elif row["IVU"]  >= 60 and row["IVEC"] < 40:  return "Potencial Urbanístico"
    elif row["IVEC"] >= 60 and row["IVU"]  < 40:  return "Potencial Económico"
    else:                                          return "Equilibrado"

df_v3["perfil_municipio"] = df_v3.apply(clasificar_perfil, axis=1)

print(df_v3["perfil_municipio"].value_counts().to_string())

## 12. Tabla resumen de resultados

In [ ]:
COLS_RESUMEN = ["municipio", "poblacion_2025", "coopscore_v6", "IVU", "IVEC", "perfil_municipio"]

display(
    df_v3[COLS_RESUMEN]
    .sort_values("coopscore_v6", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

## 13. Guardar dataset enriquecido

In [ ]:
output_path = DATA_PROCESSED / "dataset_final_tfm_v7.csv"
df_v3.to_csv(output_path, index=False)
print(f"Guardado: {output_path}  ({df_v3.shape[0]} filas, {df_v3.shape[1]} columnas)")

## 14. Importancia de variables — Random Forest + SHAP

> Sección exploratoria: entrena un RF sobre `coopscore_v6` para auditar qué features dominan el score.

In [ ]:
FEATURES = [
    "poblacion_2025",
    "superficie_residencial_m2",
    "n_poligonos",
    "suelo_por_habitante",
    "distancia_capital_km",
    "score_litoral",
]

X = df_v3[FEATURES]
y = df_v3["coopscore_v6"]

modelo_v6 = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
modelo_v6.fit(X, y)

importancias = (
    pd.DataFrame({"variable": FEATURES, "importancia": modelo_v6.feature_importances_})
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)
print(importancias.to_string(index=False))

In [ ]:
explainer   = shap.TreeExplainer(modelo_v6)
shap_values = explainer.shap_values(X)

shap.summary_plot(shap_values, X)

In [ ]:
def waterfall_municipio(nombre: str):
    """Muestra el gráfico SHAP waterfall para un municipio concreto."""
    idx = df_v3.index[df_v3["municipio"] == nombre]
    if len(idx) == 0:
        print(f"Municipio '{nombre}' no encontrado.")
        return
    i = idx[0]
    shap.plots.waterfall(
        shap.Explanation(
            values       = shap_values[i],
            base_values  = explainer.expected_value,
            data         = X.iloc[i],
            feature_names= X.columns,
        )
    )

for municipio in ["Gabias, Las", "Benamaurel", "Almuñécar"]:
    waterfall_municipio(municipio)